In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("test").remote("sc://spark-connector-0-driver-svc:15002").getOrCreate()

print("session created")

df_test = spark.createDataFrame([["test", 1]], ['name', 'num'])
print("Initialized test data")


session created
Initialized test data


In [11]:
from datetime import datetime

In [12]:
#
# AWS
#
discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
aws_namespace = "aws.acceptancetests"
aws_fqn = f"{aws_namespace}.smoketesttable_" + discriminator
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {aws_namespace}")
print(f"Created namespace {aws_namespace}")

df_test.writeTo(aws_fqn).create()
print(f"Wrote {aws_fqn} to AWS")

spark.sql(f"SELECT * FROM {aws_fqn}").show()


Created namespace aws.acceptancetests
Wrote aws.acceptancetests.smoketesttable22July2026_08_43 to AWS
+----+---+
|name|num|
+----+---+
|test|  1|
+----+---+



In [13]:
#
# GCP
#
discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
gcp_namespace = "gcp.acceptancetests"
gcp_fqn = f"{gcp_namespace}.smoketesttable_" + discriminator
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {gcp_namespace}")
print(f"Created namespace {gcp_namespace}")

df_test.writeTo(gcp_fqn).create()
print(f"Wrote {gcp_fqn} to GCP")

df_read = spark.read.table(gcp_fqn)
df_read.show()


Created namespace gcp.acceptancetests
Wrote gcp.acceptancetests.smoketesttable_22July2026_08_44 to GCP
+----+---+
|name|num|
+----+---+
|test|  1|
+----+---+



In [ ]:
#
# AZURE
#
discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
az_namespace = "azure.acceptancetests"
az_fqn = f"{az_namespace}.smoketesttable_" + discriminator

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {az_namespace}")
print(f"Created namespace {az_namespace}")

spark.sql(f"CREATE TABLE IF NOT EXISTS {az_fqn} (name String, num Integer) USING iceberg")
df_test.writeTo(az_fqn).append()

df_read = spark.read.table(az_fqn)
df_read.show()



Created namespace azure.acceptancetests


In [10]:

df_gcp = spark.read.table(gcp_fqn)
df_azure = spark.read.table(az_fqn)
df_aws = spark.read.table(aws_fqn)

df_joined = (
    df_gcp
    .join(df_azure, on="name", how="inner")
    .join(df_aws, on="name", how="inner")
)
df_joined.show()


+----+---+---+---+
|name|num|num|num|
+----+---+---+---+
|test|  1|  1|  1|
+----+---+---+---+



NameError: name 'df_glue' is not defined